Ячейка 2 — Setup & Config

In [1]:
# %%
"""
SETUP & CONFIG
"""
import json
import re
import logging
from pathlib import Path
from collections import defaultdict, Counter
from typing import Any

RAW = Path("../data/ncs_raw_json")
OUT = Path("../data/structured_data")

for sub in ["compositions", "names", "parts", "pools", "sources", "dlc", "bosses", "_meta"]:
    (OUT / sub).mkdir(parents=True, exist_ok=True)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-7s | %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger("structured")

MANUS = {"BOR", "DAD", "JAK", "MAL", "ORD", "TED", "TOR", "VLA"}
WTYPES = {"PS", "SR", "AR", "SG", "SM", "HW"}

ELEMENT_PARTS = {
    "part_element_fire": "Fire", "part_fire": "Fire", "pearl_fire": "Fire",
    "part_element_cryo": "Cryo", "part_cryo": "Cryo", "pearl_cryo": "Cryo",
    "part_element_shock": "Shock", "part_shock": "Shock", "pearl_shock": "Shock",
    "part_element_radiation": "Radiation", "part_radiation": "Radiation", "pearl_radiation": "Radiation",
    "part_element_corrosive": "Corrosive", "part_corrosive": "Corrosive", "pearl_corrosive": "Corrosive",
    "pearl_normal": "Kinetic", "part_kinetic": "Kinetic",
}

COMP_KEY_RE = re.compile(r"^comp_0(\d)_(.+)$", re.I)
TIER_ONLY_RE = re.compile(
    r"^comp_0[1-6]$|"
    r"^comp_0[1-6]_(common|uncommon|rare|epic|legendary|pearlescent|pearl)$",
    re.I,
)
KEY_FLAVOR_RE = re.compile(r"^uistat_(.+?)_(red_text|desc)$", re.I)
GUID_TRIPLE = re.compile(r"^(.+?),\s*([0-9A-Fa-f]{32}),\s*(.+)$", re.S)
HANDLE_RE = re.compile(r"inv'([^']+)'", re.I)

STRONG_DLC_MARKERS = (
    "nightmare", "glidepack", "raid1", "raid2", "raid_",
    "cello", "banjo", "cowbell", "tuba", "harp", "mandolin",
)

REGION_PATTERNS = [
    ("Fadefields", re.compile(r"\bfadefields\b", re.I)),
    ("Carcadia Burn", re.compile(r"carcadia\s*burn|\bcarcadia\b", re.I)),
    ("Terminus Range", re.compile(r"terminus\s*range|\bterminus\b", re.I)),
    ("Dominion", re.compile(r"\bdominion\b", re.I)),
    ("Whispering Glacier", re.compile(r"whispering\s*glacier|vault of the damned", re.I)),
]
KILL_RE = re.compile(
    r"Kill\s+(.+?)\s+in\s+UVH\s+(\d+)\s*\(([^)]+)\)",
    re.I | re.S,
)

log.info("Setup complete. RAW = %s", RAW.resolve())

22:57:53 | INFO    | Setup complete. RAW = /var/home/nexpg/RawBL4ToCutBDInterpreter/data/ncs_raw_json


Ячейка 3 — Utility functions

In [2]:
# %%
"""
UTILITY FUNCTIONS
"""
def load(p: Path) -> Any:
    with open(p, encoding="utf-8") as f:
        return json.load(f)

def is_named_comp(name: str) -> bool:
    if not name or not isinstance(name, str):
        return False
    return bool(COMP_KEY_RE.match(name)) and not TIER_ONLY_RE.match(name)

def rarity_from_comp(name: str) -> str:
    m = COMP_KEY_RE.match(name)
    if not m:
        return "unknown"
    digit, rest = m.group(1), m.group(2).lower()
    if digit == "6" or "pearl" in rest:
        return "pearlescent"
    if digit == "5" or "legend" in rest:
        return "legendary"
    return {"1": "common", "2": "uncommon", "3": "rare", "4": "epic"}.get(digit, f"tier_{digit}")

def internal_name(comp: str) -> str:
    m = COMP_KEY_RE.match(comp)
    if not m:
        return comp
    return re.sub(r"^(legendary_|pearl(?:escent)?_)", "", m.group(2), flags=re.I)

def _num(x) -> int | None:
    if x is None:
        return None
    try:
        return int(float(str(x)))
    except Exception:
        return None

def extract_part_rules(value: dict) -> dict:
    rules = {}
    pairs = ((value or {}).get("parttypeselectionrules") or {}).get("pairs") or {}
    for pair in pairs.values():
        if not isinstance(pair, dict):
            continue
        slot = pair.get("key")
        val = pair.get("value") or {}
        if not slot:
            continue
        parts = []
        for p in val.get("parts") or []:
            if isinstance(p, dict) and p.get("part"):
                parts.append(p["part"])
            elif isinstance(p, str):
                parts.append(p)
        pc = val.get("partcount") or {}
        rules[slot] = {
            "parts": parts,
            "min": _num(pc.get("min")),
            "max": _num(pc.get("max")),
        }
    return rules

def extract_basetags(value: dict) -> dict:
    tags = {}
    for item in (value or {}).get("basetags") or []:
        if isinstance(item, dict):
            tags.update(item)
    return tags

def extract_uni(tags: dict) -> str | None:
    for k in tags:
        if k.lower().startswith("uni_"):
            return k
    return None

def elements_from_rules(rules: dict) -> tuple[list, str]:
    found = []
    for info in rules.values():
        for p in info.get("parts") or []:
            pl = p.lower()
            if pl in ELEMENT_PARTS:
                found.append(ELEMENT_PARTS[pl])
    found = sorted(set(found))
    if len(found) == 1:
        return found, "fixed"
    if len(found) > 1:
        return found, "pool"
    return [], "unknown"

def walk_strings(obj, out=None):
    if out is None:
        out = []
    if isinstance(obj, str):
        out.append(obj)
    elif isinstance(obj, dict):
        for v in obj.values():
            walk_strings(v, out)
    elif isinstance(obj, list):
        for v in obj:
            walk_strings(v, out)
    return out

def parse_statvalue(sv) -> tuple | None:
    if isinstance(sv, str):
        m = GUID_TRIPLE.match(sv.strip())
        if m:
            return m.group(1).strip(), m.group(2).upper(), m.group(3).strip()
    if isinstance(sv, dict):
        for v in sv.values():
            if isinstance(v, str) and re.search(r"[0-9A-Fa-f]{32}", v):
                r = parse_statvalue(v)
                if r:
                    return r
        blob = json.dumps(sv, ensure_ascii=False)
        m = re.search(r"([A-Za-z0-9_]+),\s*([0-9A-Fa-f]{32}),\s*(.+?)(?:\"|$)", blob)
        if m:
            return m.group(1), m.group(2).upper(), m.group(3).rstrip('\\", ')
    return None

def parse_name_triple(partname: str) -> tuple[str | None, str | None, str | None]:
    """category, guid, display_name — или (None, None, None)"""
    if not isinstance(partname, str):
        return None, None, None
    m = GUID_TRIPLE.match(partname.strip())
    if not m:
        return None, None, None
    cat, guid, text = m.group(1).strip(), m.group(2).upper(), m.group(3).strip()
    if len(guid) != 32:
        return None, None, None
    return cat, guid, text

log.info("Utility functions loaded")

22:57:53 | INFO    | Utility functions loaded


Ячейка 4 — Pass A: Compositions

In [3]:
# %%
"""
PASS A — COMPOSITIONS
"""
log.info("=== PASS A: compositions ===")

inv_files = sorted(
    p for p in RAW.rglob("*.json")
    if p.name.lower().startswith("inv") and "name" not in p.name.lower()
)
log.info("inv files: %d", len(inv_files))

compositions: dict[str, dict] = {}
stats = Counter()

for fi, path in enumerate(inv_files, 1):
    if fi % 15 == 0:
        log.info("inv %d/%d  %s", fi, len(inv_files), path.name)

    data = load(path)
    for tbody in (data.get("tables") or {}).values():
        for rec in tbody.get("records") or []:
            for entry in rec.get("entries") or []:
                item_type = entry.get("key") or ""

                for dep in entry.get("dep_entries") or []:
                    if dep.get("dep_table_name") != "inv_comp":
                        continue

                    val = dep.get("value") if isinstance(dep.get("value"), dict) else {}
                    comp_key = dep.get("key") or ""
                    inv_comp = val.get("inv_comp") or comp_key
                    name = inv_comp if is_named_comp(str(inv_comp)) else comp_key

                    if not is_named_comp(str(name)):
                        stats["skipped"] += 1
                        continue

                    stats["named"] += 1
                    rules = extract_part_rules(val)
                    tags = extract_basetags(val)
                    uni = extract_uni(tags)
                    elems, emode = elements_from_rules(rules)
                    serial = val.get("serialindex") or {}
                    nk = name.lower()

                    mandatory = []
                    for slot, info in rules.items():
                        parts = info.get("parts") or []
                        mn = info.get("min")
                        if mn is not None and mn >= 1 and parts:
                            mandatory.extend(parts)
                        elif slot in ("barrel", "unique", "class_mod_body") and len(parts) == 1:
                            mandatory.extend(parts)

                    if nk not in compositions:
                        compositions[nk] = {
                            "composition": name,
                            "composition_key": comp_key,
                            "internal_name": internal_name(name),
                            "rarity": rarity_from_comp(name),
                            "kind": "other",
                            "item_types": [],
                            "uni": uni,
                            "basetags": tags,
                            "slots": rules,
                            "mandatory_parts": [],
                            "elements": elems,
                            "elements_mode": emode,
                            "serialindex": serial.get("index"),
                            "serial_status": serial.get("status"),
                            "basecomposition": val.get("basecomposition"),
                            "file_sources": [],
                            # заполним позже
                            "display_name": None,
                            "display_guid": None,
                            "display_category": None,
                            "red_text": None,
                            "legendary_effect": None,
                            "has_phosphene": False,
                            "drop_sources": [],
                            "has_world_drop": False,
                            "origin_signals": [],
                        }

                    c = compositions[nk]

                    if item_type and item_type not in c["item_types"]:
                        c["item_types"].append(item_type)

                    if uni and not c["uni"]:
                        c["uni"] = uni

                    for slot, info in rules.items():
                        if slot not in c["slots"]:
                            c["slots"][slot] = info
                        else:
                            old = set(c["slots"][slot].get("parts") or [])
                            new = set(info.get("parts") or [])
                            c["slots"][slot]["parts"] = sorted(old | new)

                    for p in mandatory:
                        if p not in c["mandatory_parts"]:
                            c["mandatory_parts"].append(p)

                    if elems and c["elements_mode"] == "unknown":
                        c["elements"] = elems
                        c["elements_mode"] = emode
                    elif emode == "fixed":
                        c["elements"] = elems
                        c["elements_mode"] = "fixed"

                    src = f"{path.name}:{item_type}"
                    if src not in c["file_sources"]:
                        c["file_sources"].append(src)

log.info("stats: %s", dict(stats))
log.info("unique named compositions: %d", len(compositions))
log.info("by rarity: %s", dict(Counter(c["rarity"] for c in compositions.values())))

22:57:53 | INFO    | === PASS A: compositions ===
22:57:53 | INFO    | inv files: 52
22:57:54 | INFO    | inv 15/52  inv_21.json
22:57:54 | INFO    | inv 30/52  inv_custom.json
22:57:54 | INFO    | inv 45/52  inv_params_2.json
22:57:54 | INFO    | stats: {'skipped': 6199, 'named': 2495}
22:57:54 | INFO    | unique named compositions: 245
22:57:54 | INFO    | by rarity: {'legendary': 238, 'uncommon': 1, 'pearlescent': 6}


Ячейка 5 — Pass B: Display Names

In [4]:
# %%
"""
PASS B — DISPLAY NAMES + display_guid
"""
log.info("=== PASS B: display names + guid ===")

display: dict[str, dict] = {}

np_files = sorted(p for p in RAW.rglob("*.json") if "inv_name_part" in p.name.lower())
for path in np_files:
    data = load(path)
    for tbody in (data.get("tables") or {}).values():
        for rec in tbody.get("records") or []:
            for entry in rec.get("entries") or []:
                key = (entry.get("key") or "").lower()
                val = entry.get("value") if isinstance(entry.get("value"), dict) else {}
                partname = val.get("partname") or ""

                category, guid, display_name = parse_name_triple(partname)

                if key:
                    display[key] = {
                        "np_key": key,
                        "display": display_name,
                        "display_guid": guid,
                        "category": category,
                        "partname_raw": partname if not guid else None,
                        "minrarity": val.get("minrarity"),
                        "maxrarity": val.get("maxrarity"),
                        "requiredmanufacturer": val.get("requiredmanufacturer"),
                    }

with_guid = sum(1 for v in display.values() if v.get("display_guid"))
log.info("display entries: %d | with guid: %d", len(display), with_guid)

22:57:54 | INFO    | === PASS B: display names + guid ===
22:57:54 | INFO    | display entries: 628 | with guid: 628


Ячейка 6 — Pass C: Flavor (red_text + effect)

In [5]:
# %%
"""
PASS C — FLAVOR (red_text + legendary_effect)
"""
log.info("=== PASS C: flavor ===")

flavor: dict[str, dict] = {}

ui_files = sorted(p for p in RAW.rglob("*.json") if p.name.lower().startswith("ui_stat"))
log.info("ui_stat files: %d", len(ui_files))

for path in ui_files:
    data = load(path)
    for tbody in (data.get("tables") or {}).values():
        for rec in tbody.get("records") or []:
            for entry in rec.get("entries") or []:
                key = entry.get("key") or ""
                m = KEY_FLAVOR_RE.match(key)
                if not m:
                    continue

                stem, kind = m.group(1), m.group(2).lower()
                val = entry.get("value") if isinstance(entry.get("value"), dict) else {}
                parsed = parse_statvalue(val.get("statvalue"))

                slot = flavor.setdefault(stem.lower(), {
                    "stem": stem,
                    "red_text": None,
                    "legendary_effect": None,
                    "sources": [],
                })

                src = f"{path.name}:{key}"
                if src not in slot["sources"]:
                    slot["sources"].append(src)

                payload = {
                    "text": parsed[2] if parsed else None,
                    "guid": parsed[1] if parsed else None,
                    "category": parsed[0] if parsed else None,
                    "ui_stat": val.get("ui_stat"),
                    "displaygroup": val.get("displaygroup"),
                    "entry_key": key,
                }

                if kind == "red_text":
                    if slot["red_text"] is None or (payload["text"] and not (slot["red_text"] or {}).get("text")):
                        slot["red_text"] = payload
                else:
                    if slot["legendary_effect"] is None or (payload["text"] and not (slot["legendary_effect"] or {}).get("text")):
                        slot["legendary_effect"] = payload

with_red = sum(1 for v in flavor.values() if v["red_text"] and v["red_text"].get("text"))
with_fx = sum(1 for v in flavor.values() if v["legendary_effect"] and v["legendary_effect"].get("text"))
log.info("flavor stems: %d | with red_text: %d | with effect: %d", len(flavor), with_red, with_fx)

22:57:54 | INFO    | === PASS C: flavor ===
22:57:54 | INFO    | ui_stat files: 22
22:57:55 | INFO    | flavor stems: 228 | with red_text: 174 | with effect: 220


Ячейка 7 — Pass D: Part pools by item type

In [6]:
# %%
"""
PASS D — PART POOLS BY ITEM TYPE
"""
log.info("=== PASS D: part pools ===")

type_pools: dict[str, dict[str, set]] = defaultdict(lambda: defaultdict(set))

for path in inv_files:
    data = load(path)
    for tbody in (data.get("tables") or {}).values():
        for rec in tbody.get("records") or []:
            for entry in rec.get("entries") or []:
                item_type = entry.get("key") or ""
                bits = item_type.upper().split("_")
                type_key = f"{bits[0]}_{bits[1]}" if (len(bits) == 2 and bits[0] in MANUS and bits[1] in WTYPES) else item_type

                for dep in entry.get("dep_entries") or []:
                    if dep.get("dep_table_name") != "inv_comp":
                        continue
                    val = dep.get("value") if isinstance(dep.get("value"), dict) else {}
                    rules = extract_part_rules(val)
                    for slot, info in rules.items():
                        for p in info.get("parts") or []:
                            type_pools[type_key][slot].add(p)

type_pools_out = {
    tk: {slot: sorted(parts) for slot, parts in slots.items()}
    for tk, slots in type_pools.items()
}
log.info("item types with part pools: %d", len(type_pools_out))

22:57:55 | INFO    | === PASS D: part pools ===
22:57:56 | INFO    | item types with part pools: 91


Ячейка 8 — Pass E: Phosphene / Shiny

In [7]:
# %%
"""
PASS E — PHOSPHENE / SHINY
"""
log.info("=== PASS E: phosphene ===")

shiny_items: set[str] = set()

pool_files_all = sorted(p for p in RAW.rglob("*.json") if "itempool" in p.name.lower())
for path in pool_files_all:
    for s in walk_strings(load(path)):
        if not isinstance(s, str) or "shiny" not in s.lower():
            continue
        m = re.search(r"0[56]_legendary_([A-Za-z0-9_]+)_shiny", s, re.I)
        if m:
            shiny_items.add(m.group(1).lower())
        m = re.search(r"0[56]_pearl(?:escent)?_([A-Za-z0-9_]+)_shiny", s, re.I)
        if m:
            shiny_items.add(m.group(1).lower())

log.info("shiny internal names: %d", len(shiny_items))

22:57:56 | INFO    | === PASS E: phosphene ===
22:57:56 | INFO    | shiny internal names: 123


Ячейка 9 — Pass F: Dedicated sources (ItemPoolList)

In [8]:
# %%
"""
PASS F — DEDICATED DROP SOURCES (ItemPoolList)
Точный handle → [{key, is_trueboss}]
"""
log.info("=== PASS F: dedicated sources ===")

dedicated_map: dict[str, list] = defaultdict(list)  # handle → list of source dicts
seen_pairs: set[tuple] = set()

list_files = sorted(p for p in RAW.rglob("*.json") if "itempoollist" in p.name.lower())
log.info("ItemPoolList files: %d", len(list_files))

for path in list_files:
    data = load(path)
    for tbody in (data.get("tables") or {}).values():
        for rec in tbody.get("records") or []:
            for entry in rec.get("entries") or []:
                pool_key = (entry.get("key") or "").strip()
                if not pool_key:
                    continue
                pk_lower = pool_key.lower()
                is_trueboss = "_trueboss" in pk_lower

                for s in walk_strings(entry):
                    if not isinstance(s, str):
                        continue
                    m = HANDLE_RE.search(s)
                    if not m:
                        continue
                    handle = m.group(1).lower()
                    pair = (handle, pk_lower)
                    if pair in seen_pairs:
                        continue
                    seen_pairs.add(pair)
                    dedicated_map[handle].append({
                        "key": pool_key,
                        "is_trueboss": is_trueboss,
                    })

log.info("handles with dedicated sources: %d", len(dedicated_map))

22:57:56 | INFO    | === PASS F: dedicated sources ===
22:57:56 | INFO    | ItemPoolList files: 22
22:57:56 | INFO    | handles with dedicated sources: 210


Ячейка 10 — Pass G: World drop map

In [9]:
# %%
"""
PASS G — WORLD DROP (itempool*)
"""
log.info("=== PASS G: world drop ===")

world_handles: set[str] = set()

pool_files = sorted(
    p for p in RAW.rglob("*.json")
    if "itempool" in p.name.lower() and "list" not in p.name.lower()
)

for path in pool_files:
    data = load(path)
    for tbody in (data.get("tables") or {}).values():
        for rec in tbody.get("records") or []:
            for entry in rec.get("entries") or []:
                for s in walk_strings(entry):
                    if isinstance(s, str):
                        m = HANDLE_RE.search(s)
                        if m:
                            world_handles.add(m.group(1).lower())

log.info("handles seen in world/itempool: %d", len(world_handles))

22:57:56 | INFO    | === PASS G: world drop ===
22:57:56 | INFO    | handles seen in world/itempool: 663


Ячейка 11 — Pass H: DLC registry

In [10]:
# %%
"""
PASS H — DLC REGISTRY (DLCDef)
Только то, что есть. Без привязки к айтемам.
"""
log.info("=== PASS H: DLC registry ===")

dlc_registry = []
seen_dlc_keys = set()

for path in sorted(RAW.rglob("DLCDef*.json")):
    data = load(path)
    for tbody in (data.get("tables") or {}).values():
        for rec in tbody.get("records") or []:
            for entry in rec.get("entries") or []:
                key = entry.get("key")
                if not key or key in seen_dlc_keys:
                    continue
                seen_dlc_keys.add(key)

                val = entry.get("value") if isinstance(entry.get("value"), dict) else {}
                title_raw = val.get("dlctitle")
                cat, guid, title = parse_name_triple(title_raw) if isinstance(title_raw, str) else (None, None, None)

                dlc_registry.append({
                    "key": key,
                    "dlcname": val.get("dlcname"),
                    "title": title,
                    "title_guid": guid,
                    "title_raw": title_raw if not title else None,
                    "dedicateddroptable": val.get("dedicateddroptable"),
                    "bisbountypack": val.get("bisbountypack"),
                    "dlcdatalayer": val.get("dlcdatalayer"),
                    "entitlementfact": val.get("entitlementfact"),
                })

log.info("DLC registry entries: %d", len(dlc_registry))
for e in dlc_registry:
    log.info("  %s | %s | %s", e["key"], e.get("dlcname"), e.get("title"))

22:57:56 | INFO    | === PASS H: DLC registry ===
22:57:56 | INFO    | DLC registry entries: 7
22:57:56 | INFO    |   dlcdef_headhunter_oak2 | headhunter | Firehawk's Fury
22:57:56 | INFO    |   dlcdef_oak2 | Oak2 | Borderlands 4
22:57:56 | INFO    |   dlcdef_premium_oak2 | premium | Ornate Order Pack
22:57:56 | INFO    |   dlcdef_preorder_oak2 | preorder | Gilded Glory Pack
22:57:56 | INFO    |   dlcdef_banjo | Banjo | Happy Mercenary Day!
22:57:56 | INFO    |   dlcdef_raid1 | Raid1 | Raid1 Title
22:57:56 | INFO    |   dlcdef_cello | Cello | Vault x Hunter


Ячейка Pass I

In [11]:
# %%
"""
PASS I — bosses
merge variants + display из NCS (bossreplay + NameData), без словарей
"""
log.info("=== PASS I: bosses ===")

VARIANT_SUFFIXES = ("trueboss", "true")
JUNK_SUBSTR = (
    "cheat_", "trait_", "lootable", "ordonite_pgg", "enemy_baseloot",
)
TABLE_KEY_RE = re.compile(r"bossreplay|replaycosts|replay_costs", re.I)

def _norm(s: str) -> str:
    return re.sub(r"[^a-z0-9]", "", (s or "").lower())

def _tokens(s: str) -> set[str]:
    return set(re.findall(r"[a-z0-9]{3,}", (s or "").lower()))

def _short(pool_key: str) -> str:
    return re.sub(r"^itempoollist_", "", pool_key, flags=re.I)

def _split_variant(short: str) -> tuple[str, str | None]:
    sl = short.lower()
    for suf in sorted(VARIANT_SUFFIXES, key=len, reverse=True):
        tail = "_" + suf
        if sl.endswith(tail):
            return sl[: -len(tail)], suf
    return sl, None

def _is_pool_row(k: str) -> bool:
    kl = k.lower()
    if not kl.startswith("itempoollist_"):
        return False
    if any(j in kl for j in JUNK_SUBSTR):
        return False
    return True

def _parse_comment(comment: str | None) -> tuple[str | None, str | None]:
    if not isinstance(comment, str):
        return None, None
    m = GUID_TRIPLE.match(comment.strip())
    if not m:
        return None, None
    return m.group(2).upper(), m.group(3).strip()

def _split_group_display(display: str) -> list[str]:
    if ":" in display:
        head, tail = display.split(":", 1)
        parts = [head.strip()]
        parts += [p.strip() for p in re.split(r",| & | and ", tail) if p.strip()]
        return [p for p in parts if p]
    parts = [p.strip() for p in re.split(r",| & | and ", display) if p.strip()]
    return parts if len(parts) > 1 else [display]

# ---------------------------------------------------------------------------
# 1) ItemPoolList keys → groups
# ---------------------------------------------------------------------------
pool_keys: set[str] = set()
for path in RAW.rglob("*.json"):
    if "itempoollist" not in path.name.lower():
        continue
    try:
        data = load(path)
    except Exception:
        continue
    for tbody in (data.get("tables") or {}).values():
        for rec in tbody.get("records") or []:
            for entry in rec.get("entries") or []:
                k = str(entry.get("key") or "")
                if _is_pool_row(k):
                    pool_keys.add(k.lower())

groups: dict[str, dict] = {}
for k in pool_keys:
    short = _short(k)
    base, variant = _split_variant(short)
    if base not in groups:
        groups[base] = {
            "boss_key": base,
            "pool_keys": set(),
            "variants": set(),
        }
    groups[base]["pool_keys"].add(k)
    groups[base]["variants"].add(variant if variant else "base")

# dedicated_map: handle → list[{key: pool, ...}]  (из Pass dedicated)
# строим boss_key → handles
boss_to_handles: dict[str, set] = defaultdict(set)
if "dedicated_map" in globals() and dedicated_map:
    for handle, lst in dedicated_map.items():
        if not isinstance(lst, list):
            continue
        for item in lst:
            if isinstance(item, dict):
                pk = str(item.get("key") or item.get("pool") or "").lower()
            else:
                pk = str(item).lower()
            if not pk.startswith("itempoollist_"):
                continue
            short = _short(pk)
            base, _ = _split_variant(short)
            boss_to_handles[base].add(str(handle))

# ---------------------------------------------------------------------------
# 2) BossReplay from gbx_ue_data_table*
# ---------------------------------------------------------------------------
by_row_lower: dict[str, dict] = {}
by_row_norm: dict[str, dict] = {}
by_disp_norm: dict[str, dict] = {}
segment_index: dict[str, dict] = {}

for path in sorted(RAW.rglob("*.json")):
    if "gbx_ue_data_table" not in path.name.lower():
        continue
    try:
        data = load(path)
    except Exception:
        continue
    for tbody in (data.get("tables") or {}).values():
        for rec in tbody.get("records") or []:
            for entry in rec.get("entries") or []:
                tkey = str(entry.get("key") or "")
                if not TABLE_KEY_RE.search(tkey):
                    continue
                val = entry.get("value")
                if not isinstance(val, dict):
                    continue
                for item in val.get("data") or []:
                    if not isinstance(item, dict):
                        continue
                    rn = item.get("row_name")
                    rv = item.get("row_value") if isinstance(item.get("row_value"), dict) else {}
                    guid, display = _parse_comment(rv.get("comment"))
                    if not rn or not display:
                        continue
                    row = {
                        "row_name": str(rn),
                        "display": display,
                        "display_guid": guid,
                        "table_key": tkey,
                        "file": path.name,
                    }
                    kl = str(rn).lower()
                    if kl not in by_row_lower:
                        by_row_lower[kl] = row
                    by_row_norm[_norm(str(rn))] = row
                    by_disp_norm[_norm(display)] = row
                    for seg in _split_group_display(display):
                        ns = _norm(seg)
                        if len(ns) >= 4 and ns not in segment_index:
                            segment_index[ns] = {
                                "display": seg,
                                "display_guid": guid,
                                "parent_row": str(rn),
                                "parent_display": display,
                            }

log.info(
    "bossreplay unique row_name: %s | segments: %s",
    len(by_row_lower),
    len(segment_index),
)

def _resolve_display(boss_key: str) -> tuple[str | None, str | None, str | None]:
    bl = boss_key.lower()
    bn = _norm(boss_key)

    if bl in by_row_lower:
        r = by_row_lower[bl]
        return r["display"], r["display_guid"], "row_exact"
    if bn in by_row_norm:
        r = by_row_norm[bn]
        return r["display"], r["display_guid"], "row_norm"
    if bn in by_disp_norm:
        r = by_disp_norm[bn]
        return r["display"], r["display_guid"], "display_eq_key"

    best, best_len = None, 0
    for kl, r in by_row_lower.items():
        kn = _norm(kl)
        if len(kn) >= 5 and kn in bn and len(kn) > best_len:
            best_len = len(kn)
            best = r
    if best and best_len >= 5:
        return best["display"], best["display_guid"], "row_substr"

    if bn in segment_index:
        s = segment_index[bn]
        if s["display"].lower() not in {"meathead", "foundry freaks", "hovercarts"}:
            return s["display"], s["display_guid"], "segment"

    bt = _tokens(boss_key)
    best, best_sc = None, 0
    for kl, r in by_row_lower.items():
        inter = bt & _tokens(kl)
        if len(inter) >= 2:
            sc = len(inter) * 10 + min(len(_norm(kl)), len(bn))
            if sc > best_sc:
                best_sc = sc
                best = r
    if best and best_sc >= 20:
        return best["display"], best["display_guid"], "token_row"

    return None, None, None

# ---------------------------------------------------------------------------
# 2b) NameData from display_data*
# ---------------------------------------------------------------------------
namedata_hits: dict[str, dict] = {}

for path in sorted(RAW.rglob("*.json")):
    if "display_data" not in path.name.lower():
        continue
    try:
        text = path.read_text(encoding="utf-8", errors="ignore")
    except Exception:
        continue
    for m in re.finditer(
        r"(NameData_[A-Za-z0-9_]+),\s*([0-9A-Fa-f]{32}),\s*([^\"\\\n]{2,80})",
        text,
    ):
        cat, guid, disp = m.group(1), m.group(2).upper(), m.group(3).strip().strip('"')
        if len(disp) < 2 or disp.endswith(","):
            continue
        namedata_hits[_norm(disp)] = {
            "display": disp,
            "display_guid": guid,
            "category": cat,
        }

def _resolve_namedata(boss_key: str) -> tuple[str | None, str | None, str | None]:
    bn = _norm(boss_key)
    if bn in namedata_hits:
        h = namedata_hits[bn]
        return h["display"], h["display_guid"], "namedata_eq"
    best, best_len = None, 0
    for nn, h in namedata_hits.items():
        if len(nn) < 5:
            continue
        if nn in bn or bn in nn:
            if min(len(nn), len(bn)) > best_len:
                best_len = min(len(nn), len(bn))
                best = h
    if best and best_len >= 6:
        return best["display"], best["display_guid"], "namedata_substr"
    return None, None, None

# ---------------------------------------------------------------------------
# 3) UVH catalog
# ---------------------------------------------------------------------------
uvh_rows: list[dict] = []
seen_uvh = set()
for path in RAW.rglob("*.json"):
    if "challenge" not in path.name.lower():
        continue
    try:
        text = path.read_text(encoding="utf-8", errors="ignore")
    except Exception:
        continue
    for m in KILL_RE.finditer(text):
        boss_text = m.group(1).strip()
        loc = m.group(3).strip()
        region = None
        for rname, rx in REGION_PATTERNS:
            if rx.search(loc):
                region = rname
                break
        sig = (boss_text, loc)
        if sig in seen_uvh:
            continue
        seen_uvh.add(sig)
        uvh_rows.append({
            "boss_text": boss_text,
            "uvh_level": m.group(2),
            "location_blob": loc,
            "region": region,
        })

uvh_by_norm = defaultdict(list)
for row in uvh_rows:
    uvh_by_norm[_norm(row["boss_text"])].append(row)

def _uvh_for_display(display: str | None, boss_key: str):
    if not display:
        return None, None
    hits = uvh_by_norm.get(_norm(display), [])
    if not hits:
        hits = uvh_by_norm.get(_norm(boss_key), [])
    if not hits:
        return None, None
    regions = sorted({h["region"] for h in hits if h.get("region")})
    locs = sorted({h["location_blob"] for h in hits if h.get("location_blob")})
    region = regions[0] if len(regions) == 1 else None
    loc = locs[0] if len(locs) == 1 else ("; ".join(locs) if locs else None)
    return region, loc

# ---------------------------------------------------------------------------
# 4) Build bosses_list
# ---------------------------------------------------------------------------
bosses_list: list[dict] = []
n_named = 0
n_region = 0
methods: Counter = Counter()

for base in sorted(groups.keys()):
    g = groups[base]
    variants = g["variants"]
    has_trueboss = "trueboss" in variants
    has_true = "true" in variants
    has_base = "base" in variants

    display, dguid, how = _resolve_display(base)
    if not display:
        display, dguid, how = _resolve_namedata(base)
    if how:
        methods[how] += 1
    if display:
        n_named += 1

    region, loc = _uvh_for_display(display, base)
    if region:
        n_region += 1

    handles = sorted(boss_to_handles.get(base, set()))

    bosses_list.append({
        "boss_key": base,
        "pool_keys": sorted(g["pool_keys"]),
        "has_base_pool": has_base,
        "has_trueboss": has_trueboss,
        "has_true": has_true,
        "variant_flags": sorted(v for v in variants if v != "base"),
        "display_name": display,
        "display_guid": dguid,
        "display_source": how,
        "region": region,
        "location_blob": loc,
        "dedicated_handles": handles,
    })

# ---------------------------------------------------------------------------
# 5) Side artifacts (all.json пишет WRITE)
# ---------------------------------------------------------------------------
bosses_dir = OUT / "bosses"
bosses_dir.mkdir(parents=True, exist_ok=True)

name_map_out = {
    r["row_name"]: {
        "display": r["display"],
        "display_guid": r["display_guid"],
        "table_key": r["table_key"],
    }
    for r in by_row_lower.values()
}
with open(bosses_dir / "bossreplay_names.json", "w", encoding="utf-8") as f:
    json.dump(name_map_out, f, ensure_ascii=False, indent=2)

with open(bosses_dir / "uvh_locations.json", "w", encoding="utf-8") as f:
    json.dump(uvh_rows, f, ensure_ascii=False, indent=2)

log.info(
    "bosses: %s | with display: %s | with region: %s | methods: %s",
    len(bosses_list),
    n_named,
    n_region,
    dict(methods),
)
crash_rows = [b for b in bosses_list if b["boss_key"] == "crash"]
log.info(
    "crash rows: %s | pool_keys=%s has_true=%s has_trueboss=%s",
    len(crash_rows),
    crash_rows[0]["pool_keys"] if crash_rows else None,
    crash_rows[0]["has_true"] if crash_rows else None,
    crash_rows[0]["has_trueboss"] if crash_rows else None,
)

22:57:56 | INFO    | === PASS I: bosses ===
22:57:56 | INFO    | bossreplay unique row_name: 50 | segments: 60
22:58:44 | INFO    | bosses: 82 | with display: 53 | with region: 9 | methods: {'row_exact': 33, 'segment': 2, 'display_eq_key': 1, 'namedata_substr': 13, 'row_substr': 4}
22:58:44 | INFO    | crash rows: 1 | pool_keys=['itempoollist_crash', 'itempoollist_crash_true'] has_true=True has_trueboss=False


Ячейка 12 — Linking + Enrichment

In [12]:
# %%
"""
LINKING + ENRICHMENT
Только подтверждённые данные.
"""
log.info("=== LINKING ===")

def match_flavor(internal: str):
    inn = internal.lower()
    if inn in flavor:
        return flavor[inn]
    for stem, fl in flavor.items():
        if stem.replace("_", "") == inn.replace("_", ""):
            return fl
    return None

def origin_signals_from_sources(sources: list) -> list:
    signals = set()
    for s in sources:
        k = (s.get("key") or "").lower()
        for marker in STRONG_DLC_MARKERS:
            if marker in k:
                signals.add(marker)
    return sorted(signals)

attached_display = 0
attached_guid = 0
attached_flavor = 0
with_sources = 0
with_world = 0

for c in compositions.values():
    # --- display + guid ---
    disp = guid = cat = None
    candidates = []
    if c.get("uni"):
        u = c["uni"]
        nk = "np_" + u[4:].lower() if u.lower().startswith("uni_") else "np_" + u.lower()
        candidates.append(nk)
    candidates.append("np_" + c["internal_name"].lower())

    for nk in candidates:
        if nk in display:
            d = display[nk]
            disp = d.get("display")
            guid = d.get("display_guid")
            cat = d.get("category")
            if disp or guid:
                break

    c["display_name"] = disp
    c["display_guid"] = guid
    c["display_category"] = cat
    if disp:
        attached_display += 1
    if guid:
        attached_guid += 1

    # --- flavor ---
    fl = match_flavor(c["internal_name"])
    if fl:
        c["red_text"] = fl.get("red_text")
        c["legendary_effect"] = fl.get("legendary_effect")
        if fl.get("red_text") and fl["red_text"].get("text"):
            attached_flavor += 1
    else:
        c["red_text"] = None
        c["legendary_effect"] = None

    # --- phosphene ---
    inn = c["internal_name"].lower()
    c["has_phosphene"] = inn in shiny_items or inn.replace("_", "") in {x.replace("_", "") for x in shiny_items}

    # --- drop sources (точный handle = composition lower) ---
    handle = c["composition"].lower()
    # также пробуем варианты с item_type префиксом, если composition короткий
    srcs = list(dedicated_map.get(handle, []))
    # доп. поиск: handles вида manu_type.comp_...
    if not srcs:
        for h, lst in dedicated_map.items():
            if h.endswith("." + handle) or h.endswith(handle):
                for item in lst:
                    if item not in srcs:
                        srcs.append(item)
    c["drop_sources"] = srcs
    if srcs:
        with_sources += 1

    # --- world drop ---
    c["has_world_drop"] = handle in world_handles or any(
        h.endswith(handle) or handle in h for h in world_handles
    )
    if c["has_world_drop"]:
        with_world += 1

    # --- origin signals (только сильные маркеры, без выдумок) ---
    c["origin_signals"] = origin_signals_from_sources(srcs)

log.info("display_name: %d | display_guid: %d | red_text: %d", attached_display, attached_guid, attached_flavor)
log.info("with drop_sources: %d | with world_drop: %d", with_sources, with_world)

22:58:44 | INFO    | === LINKING ===
22:58:44 | INFO    | display_name: 0 | display_guid: 0 | red_text: 173
22:58:44 | INFO    | with drop_sources: 192 | with world_drop: 196


Ячейка 13 — Write files

In [13]:
# %%
"""
WRITE STRUCTURED FILES
(только после Linking)
"""
log.info("=== WRITE ===")

comp_list = sorted(
    compositions.values(),
    key=lambda x: (x.get("rarity", ""), x["composition"].lower())
)

with open(OUT / "compositions" / "all.json", "w", encoding="utf-8") as f:
    json.dump(comp_list, f, ensure_ascii=False, indent=2)

with open(OUT / "names" / "display_names.json", "w", encoding="utf-8") as f:
    json.dump(display, f, ensure_ascii=False, indent=2)

with open(OUT / "names" / "flavor.json", "w", encoding="utf-8") as f:
    json.dump(flavor, f, ensure_ascii=False, indent=2)

by_guid = {}
for stem, v in flavor.items():
    for field in ("red_text", "legendary_effect"):
        block = v.get(field) or {}
        g = block.get("guid")
        if g:
            by_guid[g] = {"stem": stem, "field": field, **block}
with open(OUT / "names" / "flavor_by_guid.json", "w", encoding="utf-8") as f:
    json.dump(by_guid, f, ensure_ascii=False, indent=2)

with open(OUT / "parts" / "by_item_type.json", "w", encoding="utf-8") as f:
    json.dump(type_pools_out, f, ensure_ascii=False, indent=2)

with open(OUT / "pools" / "shiny_names.json", "w", encoding="utf-8") as f:
    json.dump(sorted(shiny_items), f, ensure_ascii=False, indent=2)

sources_index = {h: lst for h, lst in dedicated_map.items()}
with open(OUT / "sources" / "dedicated_by_handle.json", "w", encoding="utf-8") as f:
    json.dump(sources_index, f, ensure_ascii=False, indent=2)

with open(OUT / "dlc" / "registry.json", "w", encoding="utf-8") as f:
    json.dump(dlc_registry, f, ensure_ascii=False, indent=2)

with open(OUT / "bosses" / "all.json", "w", encoding="utf-8") as f:
    json.dump(bosses_list, f, ensure_ascii=False, indent=2)

with open(OUT / "bosses" / "uvh_locations.json", "w", encoding="utf-8") as f:
    json.dump(uvh_rows, f, ensure_ascii=False, indent=2)

meta = {
    "named_compositions": len(comp_list),
    "by_rarity": dict(Counter(c["rarity"] for c in comp_list)),
    "with_display": sum(1 for c in comp_list if c.get("display_name")),
    "with_display_guid": sum(1 for c in comp_list if c.get("display_guid")),
    "with_red_text": sum(1 for c in comp_list if c.get("red_text") and c["red_text"].get("text")),
    "with_phosphene": sum(1 for c in comp_list if c.get("has_phosphene")),
    "with_drop_sources": sum(1 for c in comp_list if c.get("drop_sources")),
    "with_world_drop": sum(1 for c in comp_list if c.get("has_world_drop")),
    "with_origin_signals": sum(1 for c in comp_list if c.get("origin_signals")),
    "item_types_pooled": len(type_pools_out),
    "flavor_stems": len(flavor),
    "dlc_registry_count": len(dlc_registry),
    "bosses_total": len(bosses_list),
    "bosses_with_handles": sum(1 for b in bosses_list if b.get("dedicated_handles")),
    "uvh_location_rows": len(uvh_rows),
}
with open(OUT / "_meta" / "phase1_summary.json", "w", encoding="utf-8") as f:
    json.dump(meta, f, ensure_ascii=False, indent=2)

log.info("META:\n%s", json.dumps(meta, indent=2, ensure_ascii=False))
log.info("Written to %s", OUT.resolve())

22:58:44 | INFO    | === WRITE ===
22:58:44 | INFO    | META:
{
  "named_compositions": 245,
  "by_rarity": {
    "legendary": 238,
    "pearlescent": 6,
    "uncommon": 1
  },
  "with_display": 0,
  "with_display_guid": 0,
  "with_red_text": 173,
  "with_phosphene": 121,
  "with_drop_sources": 192,
  "with_world_drop": 196,
  "with_origin_signals": 22,
  "item_types_pooled": 91,
  "flavor_stems": 228,
  "dlc_registry_count": 7,
  "bosses_total": 82,
  "bosses_with_handles": 76,
  "uvh_location_rows": 21
}
22:58:44 | INFO    | Written to /var/home/nexpg/RawBL4ToCutBDInterpreter/data/structured_data


Ячейка 14 — Summary

In [14]:
# %%
"""
SUMMARY
"""
print("=" * 60)
print("STRUCTURED DATA READY")
print("=" * 60)
for k, v in meta.items():
    print(f"  {k:25} {v}")
print()
print("Files:")
for p in sorted(OUT.rglob("*.json")):
    print(" ", p.relative_to(OUT))

STRUCTURED DATA READY
  named_compositions        245
  by_rarity                 {'legendary': 238, 'pearlescent': 6, 'uncommon': 1}
  with_display              0
  with_display_guid         0
  with_red_text             173
  with_phosphene            121
  with_drop_sources         192
  with_world_drop           196
  with_origin_signals       22
  item_types_pooled         91
  flavor_stems              228
  dlc_registry_count        7
  bosses_total              82
  bosses_with_handles       76
  uvh_location_rows         21

Files:
  _meta/phase1_summary.json
  bosses/all.json
  bosses/bossreplay_names.json
  bosses/uvh_locations.json
  compositions/all.json
  dlc/registry.json
  names/display_names.json
  names/flavor.json
  names/flavor_by_guid.json
  parts/by_item_type.json
  pools/shiny_names.json
  sources/dedicated_by_handle.json
